# Experiment 5.5.2 — Progress-Anchored History-Residual Routing

**Analysis-only.** Reads finalized artifacts only; no training, alpha selection, Slurm submission, or regeneration. Coordinate → Reset → Lag1 → Ordered separates flexible progress mapping, current WHAT, previous WHAT, and accumulated history.


In [ ]:
from pathlib import Path
import json, math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    here=(start or Path.cwd()).resolve()
    for candidate in (here,*here.parents):
        if (candidate/"AGENTS.md").is_file() and (candidate/"scripts").is_dir(): return candidate
    raise FileNotFoundError("Could not find writingRing repository root")
REPO_ROOT=find_repo_root()
ARTIFACT_ROOT=REPO_ROOT/"notebooks"/"artifacts"/"experiment_5_5_2_progress_anchored_history_residual"/"progress_anchored_history_residual_v1"
FILES={"selection":"selection.json","screen":"screen_summary.csv","screen_paired":"screen_paired_deltas.csv","final":"final_summary.csv","paired":"final_paired_deltas.csv","diagnostics":"residual_diagnostics.csv","alignment":"residual_alignment.csv","probes":"residual_only_probes.csv","gaps":"generalization_gap.csv","clock_progress":"clock_progress_alignment.csv","same_what":"same_what_same_coordinate.csv","manifest":"manifest.json"}
paths={k:ARTIFACT_ROOT/v for k,v in FILES.items()}
missing=[str(p) for p in paths.values() if not p.is_file()]
if missing: raise FileNotFoundError("Exp5.5.2 finalized artifacts are incomplete:\n"+"\n".join(missing))
selection=json.loads(paths["selection"].read_text()); manifest=json.loads(paths["manifest"].read_text())
screen=pd.read_csv(paths["screen"]); screen_paired=pd.read_csv(paths["screen_paired"]); final=pd.read_csv(paths["final"]); paired=pd.read_csv(paths["paired"])
diagnostics=pd.read_csv(paths["diagnostics"]); alignment=pd.read_csv(paths["alignment"]); probes=pd.read_csv(paths["probes"])
gaps=pd.read_csv(paths["gaps"]); clock_progress=pd.read_csv(paths["clock_progress"]); same_what=pd.read_csv(paths["same_what"])
selection


## Selection and final performance

Each residual mode owns its Clock-validation-selected alpha. Oracle and test are excluded from selection.


In [ ]:
display(screen.sort_values(["residual_mode","alpha"])); display(final.sort_values("mean_test_balanced_accuracy",ascending=False))
fig,ax=plt.subplots(figsize=(8.5,5))
for mode,g in screen.groupby("residual_mode",sort=False):
    g=g.sort_values("alpha"); ax.errorbar(g["alpha"],g["mean_val_balanced_accuracy"],yerr=g["sem_val_balanced_accuracy"],marker="o",capsize=3,label=mode)
ax.set(xlabel="Residual bound alpha",ylabel="Mean validation BA",title="Clock validation screen"); ax.legend(); ax.grid(True,alpha=.25); plt.show()


## Paired mechanistic contrasts

Primary contrasts: Ordered−Anchor, Ordered−Reset, Ordered−Coordinate, Ordered−Lag1, and alignment ablations.


In [ ]:
def mean_sem(df,value,groups):
    rows=[]
    for keys,g in df.groupby(groups,sort=False):
        if not isinstance(keys,tuple): keys=(keys,)
        x=g[value].to_numpy(float); row=dict(zip(groups,keys)); row["mean"]=x.mean(); row["sem"]=0.0 if len(x)<2 else x.std(ddof=1)/math.sqrt(len(x)); rows.append(row)
    return pd.DataFrame(rows)
contrast=mean_sem(paired,"delta_test_balanced_accuracy",["anchor","comparison"]); display(contrast.sort_values(["anchor","comparison"]))
focus=contrast[contrast["comparison"].isin(["ordered_minus_anchor","ordered_minus_reset","ordered_minus_coordinate","ordered_minus_lag1","ordered_minus_circular_shift","ordered_minus_random_shuffle"])].copy()
fig,ax=plt.subplots(figsize=(9,6)); labels=focus["anchor"]+": "+focus["comparison"]; ax.errorbar(focus["mean"],np.arange(len(focus)),xerr=focus["sem"],fmt="o",capsize=3); ax.axvline(0,lw=1); ax.set_yticks(np.arange(len(focus)),labels); ax.set_xlabel("Paired test BA delta"); ax.grid(True,axis="x",alpha=.25); plt.show()


## Diagnostics, transfer, and same-WHAT examples


In [ ]:
test_diag=diagnostics[diagnostics["split"]=="test"]
display(test_diag.groupby(["anchor","residual_mode"],as_index=False).agg(q_l1=("q_l1_mean","mean"),kl=("q_kl_to_anchor_mean","mean"),weight_correction=("relative_weight_correction_mean","mean"),weight_cosine=("anchor_vs_residual_weight_cosine_mean","mean"),evidence_correction=("relative_evidence_correction_mean","mean"),saturation=("delta_z_saturation_fraction","mean"),argmax_changed=("anchor_argmax_changed_fraction","mean")))
display(mean_sem(gaps,"generalization_gap_val_minus_test_delta",["anchor","residual_mode"])); display(mean_sem(alignment,"balanced_accuracy",["anchor","ablation"])); display(clock_progress); display(mean_sem(probes[probes["split"]=="test"],"balanced_accuracy",["anchor","feature"]))
if same_what.empty: print("No pairs passed the strict same-WHAT/same-coordinate constraints.")
else:
    cols=["anchor","seed","label_a","label_b","what_cosine_similarity","what_norm_ratio","coordinate_abs_delta","anchor_q_l1_distance","delta_z_l1_distance","q_l1_distance","delta_w_pair_fro","different_top_support","margin_delta_a","margin_delta_b"]
    display(same_what[cols].head(30))


## Interpretation guardrails

- Clock+Ordered > Clock shows deployable history-residual utility, not necessarily semantic context beyond progress.
- Oracle+Ordered > Oracle+Reset and Oracle+Coordinate supports past context beyond current WHAT and scalar progress mapping.
- Oracle+Ordered > Oracle+Lag1 further supports accumulated history beyond only the previous local state.
- A drop after circularly shifting residuals supports correct temporal alignment.
- Five seeds share one fixed user-disjoint split; they are not independent split replicates.
